In [4]:
import pandas as pd
import re

# 1. Read the Excel file into a pandas DataFrame (main experiment sheet)
file_path = "master_stimuli.xlsx"
df = pd.read_excel(file_path, sheet_name=0)  # Reads the first sheet by default

# 2. Clean main data: drop rows where 'item' is missing
df_clean = df.dropna(subset=["item"]).copy()
df_clean["item"] = df_clean["item"].astype(int)

# Strip whitespace and clean NaNs from string columns to prevent matching failures & unwanted 'nan' text
string_cols = [
    "type", "cnd", "context_cnd", "target_cnd", "question", 
    "r1", "r2", "r3", "r4", "r5", "r6", "a", "b", "correct", "cb"
]
for str_col in string_cols:
    if str_col in df_clean.columns:
        df_clean[str_col] = (
            df_clean[str_col]
            .fillna("")
            .astype(str)
            .str.strip()
            .replace("nan", "")
        )

# Format r6 to add a period ONLY if it doesn't already end with '.', '?', or '!'
def smart_punctuate(text):
    if not text:
        return text
    # Check if text ends with a period, question mark, or exclamation mark
    if not re.search(r'[.?!]$', text):
        return text + '.'
    return text

if "r6" in df_clean.columns:
    df_clean["r6"] = df_clean["r6"].apply(smart_punctuate)

# Convert integer metadata columns safely
cols_to_int = ["exp", "item", "set", "vpx", "def", "past", "multi", "alt"]
for col in cols_to_int:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").astype("Int64")

# Separate targets, fillers, and checks
targets = df_clean[df_clean["type"] == "target"]
fillers = df_clean[df_clean["type"] == "filler"]
checks = df_clean[df_clean["type"] == "check"]

# Define conditions for targets (12 conditions)
target_conditions = [
    "C1T1", "C1T2", "C1T3", "C1T4",
    "C2T1", "C2T2", "C2T3", "C2T4",
    "C3T1", "C3T2", "C3T3", "C3T4"
]

# Get unique filler condition variants
filler_conditions = fillers["cnd"].unique().tolist()

# Define explicit check variants from 'cnd'
check_variants = ["attention", "attention-alt"]

print(f"Loaded targets: {len(targets)} rows ({len(target_conditions)} conditions)")
print(f"Loaded fillers: {len(fillers)} rows ({len(filler_conditions)} variants)")
print(f"Loaded checks: {len(checks)} rows ({len(check_variants)} variants)")

# 3. Process and export the 'practice' sheet
try:
    practice_df = pd.read_excel(file_path, sheet_name="practice")
    for col in practice_df.select_dtypes(include=["object"]).columns:
        practice_df[col] = (
            practice_df[col]
            .fillna("")
            .astype(str)
            .str.strip()
            .replace("nan", "")
        )
    
    if "r6" in practice_df.columns:
        practice_df["r6"] = practice_df["r6"].apply(smart_punctuate)

    practice_df.to_csv("practice.csv", index=False, encoding="utf-8-sig")
    print(f"Successfully generated: practice.csv (Total items: {len(practice_df)})")
except Exception as e:
    print(f"Warning: Could not load 'practice' sheet from Excel. Error: {e}")

# 4. Generate List 1 through List 12
for list_num in range(1, 13):
    list_target_rows = []
    list_filler_rows = []
    list_check_rows = []
    
    # --- A. Select 24 Targets (Rotated across 12 conditions) ---
    for item_id in range(1, 25):
        cnd_index = (item_id - 1 + list_num - 1) % 12
        assigned_cnd = target_conditions[cnd_index]
        row = targets[(targets["item"] == item_id) & (targets["cnd"] == assigned_cnd)]
        
        if len(row) == 0:
            print(f"Warning: Missing Target Item {item_id} with condition {assigned_cnd}")
        list_target_rows.append(row)
        
    # --- B. Select 12 Fillers (Rotated across their variants) ---
    for filler_id in range(1, 13):
        fill_index = (filler_id - 1 + list_num - 1) % len(filler_conditions)
        assigned_fill_cnd = filler_conditions[fill_index]
        row = fillers[(fillers["item"] == filler_id) & (fillers["cnd"] == assigned_fill_cnd)]
        
        if len(row) == 0:
            print(f"Warning: Missing Filler Item {filler_id} with condition {assigned_fill_cnd}")
        list_filler_rows.append(row)

    # --- C. Select 4 Attention Checks (Rotating 'cnd' between attention and attention-alt) ---
    for check_id in range(1, 5):
        variant_index = (check_id - 1 + list_num - 1) % len(check_variants)
        assigned_variant = check_variants[variant_index]
        
        row = checks[(checks["item"] == check_id) & (checks["cnd"] == assigned_variant)]
        
        if len(row) == 0:
            print(f"Warning: Missing Check Item {check_id} with cnd '{assigned_variant}'")
        list_check_rows.append(row)

    # --- D. Combine all components ---
    list_df = pd.concat(list_target_rows + list_filler_rows + list_check_rows, ignore_index=True)
    
    # Safety assertion to guarantee exactly 40 items per list
    assert len(list_df) == 40, f"Error: List {list_num} generated with {len(list_df)} items instead of 40!"
    
    # Export to CSV with UTF-8 encoding for Hungarian accents
    filename = f"list_{list_num:02d}.csv"
    list_df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"Successfully generated: {filename} (Total items: {len(list_df)})")

Loaded targets: 288 rows (12 conditions)
Loaded fillers: 48 rows (4 variants)
Loaded checks: 8 rows (2 variants)
Successfully generated: practice.csv (Total items: 3)
Successfully generated: list_01.csv (Total items: 40)
Successfully generated: list_02.csv (Total items: 40)
Successfully generated: list_03.csv (Total items: 40)
Successfully generated: list_04.csv (Total items: 40)
Successfully generated: list_05.csv (Total items: 40)
Successfully generated: list_06.csv (Total items: 40)
Successfully generated: list_07.csv (Total items: 40)
Successfully generated: list_08.csv (Total items: 40)
Successfully generated: list_09.csv (Total items: 40)
Successfully generated: list_10.csv (Total items: 40)
Successfully generated: list_11.csv (Total items: 40)
Successfully generated: list_12.csv (Total items: 40)
